In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,nan,+nan%,+nan%,+nan%,+nan%,+nan%,+nan%
NASDAQ Composite,^IXIC,nan,+nan%,+nan%,+nan%,+nan%,+nan%,+nan%
Apple,AAPL,nan,+nan%,+nan%,+nan%,+nan%,+nan%,+nan%
NVIDIA,NVDA,nan,+nan%,+nan%,+nan%,+nan%,+nan%,+nan%
Microsoft,MSFT,nan,+nan%,+nan%,+nan%,+nan%,+nan%,+nan%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"67,242.73",-1.92%,+0.63%,-3.58%,-2.99%,+10.57%,+10.57%
Toyota Motor,7203.T,"2,839.00",+1.18%,+0.53%,-3.63%,-0.30%,-3.91%,-3.91%
Sony Group,6758.T,"3,383.00",+1.26%,-0.76%,-3.40%,+3.30%,-5.92%,-5.92%
Mitsubishi UFJ Financial,8306.T,"3,600.00",+1.67%,+5.45%,+4.47%,+11.52%,+20.20%,+20.20%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,470.34",+0.02%,+1.88%,+4.00%,+7.74%,+9.48%,+9.48%
DBS Group,D05.SI,72.03,+1.75%,+2.56%,+4.94%,+13.02%,+18.55%,+18.55%
UOB,U11.SI,44.38,+0.91%,+0.09%,+6.45%,+13.79%,+19.01%,+19.01%
Singtel,Z74.SI,4.44,+0.23%,+1.14%,+1.37%,+3.26%,-8.83%,-8.83%
CapitaLand Ascendas REIT,A17U.SI,2.49,-1.19%,+0.00%,-0.40%,-1.97%,+2.05%,+2.05%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,366.00",-0.58%,-0.45%,-5.13%,-4.27%,+0.70%,+0.70%
ニデック (6594),6594.T,"2,608.00",-1.40%,-0.19%,-1.81%,-4.71%,+3.21%,+3.21%
未来工業 (7931),7931.T,"3,240.00",+0.62%,+0.93%,+0.78%,+6.40%,+9.31%,+9.31%
東部ネットワーク (9036),9036.T,"1,235.00",+0.16%,-4.11%,-3.36%,+3.09%,-5.07%,-5.07%
ニトリHD (9843),9843.T,"2,452.50",+2.32%,+1.09%,-0.49%,-4.83%,+1.07%,+1.07%
MRK HD (9980),9980.T,94.00,-1.05%,-2.08%,-2.08%,-2.08%,-4.08%,-4.08%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,nan,+nan%,+nan%,+nan%,+nan%,+nan%,+nan%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.94,+0.00%,+0.00%,+1.08%,+0.00%,-1.57%,-1.57%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.83,-0.70%,+1.80%,-0.70%,+1.80%,-3.41%,-3.41%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+1.89%,+1.89%,+0.00%,-3.57%,-5.26%,-5.26%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.13,+0.24%,+0.98%,+0.98%,+4.82%,+4.29%,+4.29%
First REIT (ファースト・リート),AW9U.SI,0.22,+0.00%,-2.17%,-2.17%,-2.17%,-4.26%,-4.26%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.55,-0.91%,-1.80%,+0.00%,-7.63%,-9.92%,-9.92%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+0.00%,+0.00%,+0.00%,+1.94%,+1.94%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,+0.00%,-2.17%,-2.17%,+0.00%,+2.27%,+2.27%
Medtecs International (医療用消耗品),546.SI,0.11,+0.89%,+0.00%,-0.88%,-6.61%,-8.13%,-8.13%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+0.00%,+0.00%,+2.94%,+11.11%,+11.11%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.89,+0.52%,+2.10%,+3.73%,+8.06%,+4.85%,+4.85%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.22,+0.00%,+2.52%,+0.83%,-7.58%,+4.27%,+4.27%
Riverstone,AP4.SI,0.86,+2.40%,+1.18%,+1.79%,+1.18%,-6.04%,-6.04%
ヤンジジャン・シップビルディング(BS6),BS6.SI,3.57,-0.28%,+4.69%,+2.59%,-0.83%,-8.93%,-8.93%
Sembcorp Ind (U96),U96.SI,5.51,-2.13%,-1.61%,-3.84%,-11.84%,-9.82%,-9.82%


<div style="background-color: #EBCB8B; color: #2E3440; padding: 12px; border-radius: 6px; font-weight: bold;">
</div>